##### Load Ground Truth

In [1]:
from typing import TypedDict, cast

import pandas as pd

from lib.types import FAQGroundTruthRecord


CSV_ENCODING = "utf-8-sig"


faq_ground_truth_df = pd.read_csv("data/faq_ground_truth.csv", encoding=CSV_ENCODING)

display(faq_ground_truth_df.head())

faq_ground_truth: list[FAQGroundTruthRecord] = cast(
    list[FAQGroundTruthRecord],
    faq_ground_truth_df.to_dict(orient="records"))


,question,document
0,I just found this course late ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â can I still st...,74eb249bbf
1,Is it too late to enroll if I discovered the c...,74eb249bbf
2,Can I still take part in the course even thoug...,74eb249bbf
3,"If I join the course late, am I still eligible...",74eb249bbf
4,WhatÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢s the deadline for submitting the project...,74eb249bbf


##### Load Original FAQ

In [2]:
from pydash import filter_

from lib.sources import load_faq_documents


llm_zoomcamp_course = {"course": "llm-zoomcamp"}

all_faq_documents = load_faq_documents()

faq_documents = filter_(all_faq_documents, llm_zoomcamp_course)

# Create a lookup table for the documents
documents_by_id = {
    doc["id"]: doc
    for doc in faq_documents
}

##### Create Agent System

In [3]:
from dotenv import dotenv_values
from openai import OpenAI

from lib.agentic_rag import AgenticRAG
from lib.search import MinsearchLexicalSearchTool
from lib.search_types import LexicalSearchConfig


INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""


config = dotenv_values()
openai_client = OpenAI(api_key=config["OPENAI_API_KEY"])

minsearch_lexical_search_tool = MinsearchLexicalSearchTool.from_documents(
    faq_documents,
    text_fields=["question", "answer"],
    keyword_fields=["course"],
    config=LexicalSearchConfig(
        num_results=5,
        boost_dict={
            "question": 1.0,
            "answer": 3.0,
        }
    )
)

agentic_rag = AgenticRAG(
    llm_client=openai_client,
    search_tool=minsearch_lexical_search_tool,
    instructions=INSTRUCTIONS,
    mode="simple",
)

##### Build Judge Cases

In [4]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

from evaluation_utils import map_progress
from lib.metrics import AgentRunMetrics, ModelCallMetrics


class RAGJudgeCase(TypedDict):
    question: str
    rag_answer: str
    original_answer: str
    document_id: str


def build_rag_judge_case(
        ground_truth_record: FAQGroundTruthRecord
        ) -> tuple[RAGJudgeCase, AgentRunMetrics]:
    question = ground_truth_record["question"]
    document_id = ground_truth_record["document"]
    original_document = documents_by_id[document_id]

    result = agentic_rag.find_and_reply(question)
    original_answer = original_document["answer"]

    judge_case: RAGJudgeCase = {
        "question": question,
        "rag_answer": result.answer,
        "original_answer": original_answer,
        "document_id": document_id
    }
    return judge_case, result.metrics


judge_cases_path = Path("data/rag_judge_cases.csv")
should_rebuild_judge_cases = False

if should_rebuild_judge_cases:

    with ThreadPoolExecutor(max_workers=3) as pool:
        rag_results = map_progress(
            pool,
            faq_ground_truth,
            build_rag_judge_case,
            submit_delay_seconds=0.5,
        )

    rag_judge_cases = [case for case, _metrics in rag_results]
    run_metrics = [metrics for _case, metrics in rag_results]
    print(sum(metrics.total_cost_usd for metrics in run_metrics))

    judge_cases_df = pd.DataFrame(rag_judge_cases)
    judge_cases_df.to_csv(judge_cases_path, index=False, encoding=CSV_ENCODING)


if judge_cases_path.exists():
    judge_cases_df = pd.read_csv(judge_cases_path, encoding=CSV_ENCODING)

else:
    judge_cases_df = pd.DataFrame(
        columns=[
            "question",
            "rag_answer",
            "original_answer",
            "document_id",
        ]
    )

rag_judge_cases: list[RAGJudgeCase] = cast(
    list[RAGJudgeCase],
    judge_cases_df.to_dict(orient="records"),
)



##### Judge Decisions

In [14]:
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, Field

from lib.llm import call_structured_llm


class RAGJudgeDecision(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )


class RAGJudgeDecisionDict(TypedDict):
    reasoning: str
    score: Literal["good", "bad"]


class RAGJudgedCase(TypedDict):
    question: str
    rag_answer: str
    original_answer: str
    document_id: str
    reasoning: str
    score: Literal["good", "bad"]



rag_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by the RAG system

Your task is to decide if the RAG answer is semantically equivalent to
the original answer.

Rules:
- The RAG answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the RAG answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

rag_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{original_answer}

RAG Answer:
{rag_answer}
""".strip()


def judge_rag_case(
    rag_judge_case: RAGJudgeCase,
) -> tuple[RAGJudgedCase, ModelCallMetrics]:
    prompt = rag_judge_prompt.format(
        question=rag_judge_case["question"],
        original_answer=rag_judge_case["original_answer"],
        rag_answer=rag_judge_case["rag_answer"],
    )

    call = call_structured_llm(
        client=openai_client,
        instructions=rag_judge_instructions,
        user_prompt=prompt,
        output_type=RAGJudgeDecision,
    )

    decision_dict = cast(
        RAGJudgeDecisionDict, call.result.model_dump()
    )

    judged_case: RAGJudgedCase = {
        **rag_judge_case,
        **decision_dict,
    }

    return judged_case, call.metrics


judged_cases_path = Path("data/rag_judged_cases.csv")
should_rebuild_judged_cases = False

if should_rebuild_judged_cases:
    with ThreadPoolExecutor(max_workers=3) as pool:
        judge_results = map_progress(
            pool,
            rag_judge_cases,
            judge_rag_case,
            submit_delay_seconds=0.5,
        )

    rag_judged_cases = [
        judged_case for judged_case, _metrics in judge_results
    ]
    judge_call_metrics = [
        metrics for _judged_case, metrics in judge_results
    ]

    print(
        sum(metrics.price.total_cost_usd for metrics in judge_call_metrics)
    )

    rag_judged_cases_df = pd.DataFrame(rag_judged_cases)
    rag_judged_cases_df.to_csv(judged_cases_path, index=False, encoding=CSV_ENCODING)


if judged_cases_path.exists():
    rag_judged_cases_df = pd.read_csv(judged_cases_path, encoding=CSV_ENCODING)

else:
    rag_judged_cases_df = pd.DataFrame(
        columns=[
            "question",
            "rag_answer",
            "original_answer",
            "document_id",
            "reasoning",
            "score",
        ]
    )

rag_judged_cases: list[RAGJudgedCase] = cast(
    list[RAGJudgedCase],
    rag_judged_cases_df.to_dict(orient="records"),
)



  0%|          | 0/425 [00:00<?, ?it/s]

0.28954350000000034


In [ ]:
score_breakdown = (
    rag_judged_cases_df["score"]
    .value_counts()
    .rename_axis("score")
    .reset_index(name="count")
)

display(score_breakdown)

bad_judged_cases_df = rag_judged_cases_df[
    rag_judged_cases_df["score"] == "bad"
]

display(
    bad_judged_cases_df[
        [
            "question",
            "original_answer",
            "rag_answer",
            "reasoning",
            "document_id",
        ]
    ].head(5)
)
